# **PySpark MLlib – Linear Regression**

### Boston Housing Dataset
Predicting **house price (`medv`)** using PySpark MLlib.

| Column    | Description                                                                                |
| --------- | ------------------------------------------------------------------------------------------ |
| `crim`    | Per capita crime rate by town                                                              |
| `zn`      | Proportion of residential land zoned for large lots                                        |
| `indus`   | Proportion of non-retail business acres                                                    |
| `chas`    | Charles River dummy variable (1 if tract bounds the river, otherwise 0)                    |
| `nox`     | Nitric oxide concentration                                                                 |
| `rm`      | Average number of rooms per dwelling                                                       |
| `age`     | Proportion of owner-occupied units built before 1940                                       |
| `dis`     | Weighted distance to employment centers                                                    |
| `rad`     | Index of accessibility to radial highways                                                  |
| `tax`     | Property-tax rate                                                                          |
| `ptratio` | Pupil-teacher ratio by town                                                                |
| `black`   | A transformed measure related to the proportion of Black residents in the original dataset |
| `lstat`   | Percentage of lower-status population                                                      |
| `medv`    | **Median value of owner-occupied homes — Target variable**                                 |


We have information about different characteristics of a neighborhood, such as crime rate, number of rooms, highway accessibility and tax rate. Our goal is to use these characteristics to predict the median house value.

## 1. Import Libraries and Create SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("Boston Housing Regression") \
    .master("local[*]") \
    .getOrCreate()

## 2. Load the Dataset

`medv` is our **target variable** (house price).

In [ ]:
df = spark.read.csv("Boston.csv",header=True,inferSchema=True)

In [ ]:
df.show(5)

In [ ]:
df.printSchema()

## 3. Select Features and Target

We will use the housing-related columns as input features.

`medv` → Target / Label

We remove `Unnamed: 0` because it is only a row index.

In [ ]:
feature_columns = ["crim", "zn", "indus", "chas", "nox", "rm",
    "age", "dis", "rad", "tax", "ptratio", "black", "lstat"]

In [ ]:
data = df.select(feature_columns + ["medv"])
data.show(5)

## 4. Convert Features into a Single Vector

MLlib expects input features in one vector column.

In [ ]:
assembler = VectorAssembler(inputCols=feature_columns,outputCol="features")


In [ ]:
data = assembler.transform(data)
data.show(5)

In [ ]:
data.select("features", "medv").show(5, truncate=False)

## 5. Prepare Training and Testing Data

80% → Training

20% → Testing

In [ ]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

## 6. Create the Linear Regression Model

In [ ]:
lr = LinearRegression(featuresCol="features",labelCol="medv")

## 7. Train the Model

In [ ]:
model = lr.fit(train_data)

print("Model trained successfully!")

## 8. Make Predictions

In [ ]:
predictions = model.transform(test_data)

predictions.select("medv","prediction").show(10)

## 9. Evaluate the Model

We use **RMSE** and **R²**.

In [ ]:
evaluator_rmse = RegressionEvaluator(
    labelCol="medv",
    predictionCol="prediction",
    metricName="rmse")

evaluator_r2 = RegressionEvaluator(
    labelCol="medv",
    predictionCol="prediction",
    metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("RMSE:", rmse)
print("R²:", r2)

## 10. View Model Coefficients

The model learns a coefficient for each feature.

In [ ]:
print("Intercept:", model.intercept)

for feature, coefficient in zip(feature_columns, model.coefficients):
    print(feature, ":", coefficient)

**MLlib workflow:**

CSV → DataFrame → Features + Label → VectorAssembler → Train/Test Split → Linear Regression → Prediction → Evaluation

### Important terms

- **Features:** Input variables used to make predictions.
- **Label:** Target variable we want to predict.
- **VectorAssembler:** Combines features into one vector.
- **LinearRegression:** Machine Learning algorithm.
- **RMSE:** Measures prediction error; lower is generally better.
- **R²:** Measures how well the model explains variation in the target; closer to 1 is generally better.

In [ ]:
spark.stop()